# F.I.X. S001 Colab training and export

Open in Google Colab and run **Dependency setup** first. If it installs packages, choose
**Runtime > Restart session**, then **Run all**. Wait for the import check to pass before uploading
the original retail CSV. The current LightGBM recipe runs on Colab CPU.

This notebook reuses the frozen v1 feature, label, split and calibration recipe. The dated splits
are for the original dataset. Change them explicitly before training on newer data and keep
training labels before validation origins, and validation labels before test origins.

The resulting archive contains a candidate bundle and reports. Keep the existing deployed bundle
until candidate acceptance. Daily and Monthly diagnostics are versioned rules outside training.

In [ ]:
# Dependency setup
from importlib.metadata import PackageNotFoundError, version
import os
import subprocess
import sys

pins = {
    "numpy": "2.1.3", "scipy": "1.15.3", "pandas": "2.2.3",
    "scikit-learn": "1.6.1", "joblib": "1.5.2", "lightgbm": "4.6.0",
}

def installed(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return None

restart_key = "FIX_COLAB_DEPENDENCIES_CHANGED"
restart_message = "In Colab choose Runtime > Restart session, then Run all."
if os.environ.get(restart_key) == str(os.getpid()):
    raise RuntimeError(restart_message)

if any(installed(name) != expected for name, expected in pins.items()):
    os.environ[restart_key] = str(os.getpid())
    subprocess.run([sys.executable, "-m", "pip", "install",
                    *(f"{name}=={value}" for name, value in pins.items())], check=True)
    raise RuntimeError(f"Dependencies installed. {restart_message}")

try:
    import numpy
    import scipy
    import pandas
    import sklearn
    import joblib
    import lightgbm
    modules = (numpy, scipy, pandas, sklearn, joblib, lightgbm)
    for module, (name, expected) in zip(modules, pins.items()):
        if module.__version__ != expected:
            raise RuntimeError(f"{name} loaded {module.__version__}, expected {expected}")
except (ImportError, AttributeError, ValueError, RuntimeError) as exc:
    raise RuntimeError(f"Dependency import check failed. {restart_message}") from exc

print("Dependency imports OK:", pins)

# Data Cleaning

In [ ]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
from google.colab import files
from IPython.display import display

MODEL_VERSION = "s001-global-q-colab-v1"
SOURCE_FILENAME = "retail_store_inventory_data.csv"
PROJECT_ROOT = Path("/content/fix-training") / MODEL_VERSION
if PROJECT_ROOT.exists():
    raise FileExistsError("Choose a new MODEL_VERSION or restart with a fresh Colab runtime.")

uploaded = files.upload()
if SOURCE_FILENAME not in uploaded:
    raise ValueError(f"Upload {SOURCE_FILENAME} or update SOURCE_FILENAME.")
PROJECT_ROOT.mkdir(parents=True)
RAW_PATH = PROJECT_ROOT / "source.csv"
RAW_PATH.write_bytes(uploaded[SOURCE_FILENAME])
del uploaded

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SPLITS_DIR = PROCESSED_DIR / "splits"
METADATA_DIR = PROJECT_ROOT / "data" / "metadata"
for directory in (SPLITS_DIR, METADATA_DIR):
    directory.mkdir(parents=True, exist_ok=True)

STORE_ID = "S001"
DATA_VERSION = "v1"

In [ ]:
raw = pd.read_csv(RAW_PATH, low_memory=False)

required_raw = {
    "Date",
    "Store ID",
    "Product ID",
    "Category",
    "Units Sold",
    "Units Ordered",
}

missing_columns = required_raw - set(raw.columns)

if missing_columns:
    raise ValueError(
        f"Missing CSV columns: {sorted(missing_columns)}"
    )

s001_raw = raw.loc[
    raw["Store ID"].eq(STORE_ID)
].copy()

if s001_raw.empty:
    raise ValueError("S001 not found")

print(f"Full CSV: {raw.shape}")
print(f"S001: {s001_raw.shape}")
print("Products:", s001_raw["Product ID"].nunique())
print(
    "Dates:",
    s001_raw["Date"].min(),
    "to",
    s001_raw["Date"].max()
)

display(s001_raw.head())

In [ ]:
s001_raw["Date"] = pd.to_datetime(
    s001_raw["Date"],
    errors="raise"
)

key_cols = ["Store ID", "Product ID", "Date"]

if s001_raw[key_cols].isna().any().any():
    raise ValueError("Missing identifiers")

for column in ["Units Sold", "Units Ordered"]:
    s001_raw[column] = pd.to_numeric(
        s001_raw[column],
        errors="raise"
    )

    if (
        s001_raw[column].isna().any()
        or (s001_raw[column] < 0).any()
    ):
        raise ValueError(f"Invalid values in {column}")

    if (s001_raw[column] % 1 != 0).any():
        raise ValueError(f"Non-integer quantities in {column}")

    s001_raw[column] = (
        s001_raw[column].astype("int64")
    )

if s001_raw.duplicated(key_cols).any():
    raise ValueError("Duplicate product-day records")

product_ids = sorted(
    s001_raw["Product ID"].unique().tolist()
)

all_dates = pd.date_range(
    s001_raw["Date"].min(),
    s001_raw["Date"].max(),
    freq="D"
)

expected_index = pd.MultiIndex.from_product(
    [[STORE_ID], product_ids, all_dates],
    names=key_cols
)

observed_index = pd.MultiIndex.from_frame(
    s001_raw[key_cols]
)

missing_keys = expected_index.difference(observed_index)
extra_keys = observed_index.difference(expected_index)

if len(missing_keys) or len(extra_keys):
    raise ValueError(
        f"Incomplete panel: missing={len(missing_keys)}, "
        f"extra={len(extra_keys)}"
    )

print("Products:", len(product_ids))
print("Calendar days:", len(all_dates))
print("Missing SKU-days:", len(missing_keys))
print("Duplicate records:", s001_raw.duplicated(key_cols).sum())
print("Zero-sales rows:", s001_raw["Units Sold"].eq(0).sum())

display(
    s001_raw.groupby("Product ID")["Category"]
    .nunique()
    .head()
)

In [ ]:
CATEGORY_LEAD_TIMES = {
    "Groceries": 3,
    "Toys": 7,
    "Clothing": 10,
    "Electronics": 14,
    "Furniture": 21,
}

category_cycle = list(CATEGORY_LEAD_TIMES)

product_catalog = pd.DataFrame({
    "store_id": STORE_ID,
    "product_id": product_ids,
    "category": [
        category_cycle[i % len(category_cycle)]
        for i in range(len(product_ids))
    ],
})

product_catalog["lead_time_days"] = (
    product_catalog["category"]
    .map(CATEGORY_LEAD_TIMES)
    .astype("int64")
)

assert not product_catalog[
    ["store_id", "product_id"]
].duplicated().any()

display(product_catalog)

In [ ]:
rename_columns = {
    "Date": "date",
    "Store ID": "store_id",
    "Product ID": "product_id",
    "Units Sold": "units_sold",
    "Units Ordered": "units_ordered",
    "Price": "price",
    "Discount": "discount",
    "Weather Condition": "weather_condition",
    "Holiday/Promotion": "holiday_promotion",
    "Competitor Pricing": "competitor_pricing",
    "Seasonality": "seasonality",
}

required_clean_cols = [
    "date",
    "store_id",
    "product_id",
    "units_sold",
    "units_ordered",
]

optional_clean_cols = [
    "price",
    "discount",
    "weather_condition",
    "holiday_promotion",
    "competitor_pricing",
    "seasonality",
]

normalized = s001_raw.rename(
    columns=rename_columns
)

retained_columns = required_clean_cols + [
    c for c in optional_clean_cols
    if c in normalized.columns
]

normalized = normalized[retained_columns].copy()

clean = normalized.merge(
    product_catalog,
    how="left",
    on=["store_id", "product_id"],
    validate="many_to_one",
)

clean = (
    clean.sort_values(
        ["store_id", "product_id", "date"]
    )
    .reset_index(drop=True)
)

if clean[
    ["category", "lead_time_days"]
].isna().any().any():
    raise ValueError("Product missing from catalog")

clean = clean[
    [
        "store_id",
        "product_id",
        "date",
        "category",
        "lead_time_days",
        "units_sold",
        "units_ordered",
    ]
    + [
        c for c in optional_clean_cols
        if c in clean.columns
    ]
]

assert len(clean) == len(s001_raw)

assert (
    clean.groupby("product_id")["category"]
    .nunique().eq(1).all()
)

assert (
    clean.groupby("product_id")["lead_time_days"]
    .nunique().eq(1).all()
)

assert {
    "Inventory Level",
    "Demand Forecast",
    "Region"
}.isdisjoint(clean.columns)

display(clean.head(8))

In [ ]:
CATALOG_PATH = (
    METADATA_DIR / "product_catalog_v1.csv"
)

CLEAN_PATH = (
    PROCESSED_DIR / "s001_sales_clean_v1.csv"
)

product_catalog.to_csv(
    CATALOG_PATH,
    index=False
)

clean.to_csv(
    CLEAN_PATH,
    index=False,
    date_format="%Y-%m-%d",
    float_format="%.12g",
)

# Verify that saved data can be loaded again
reloaded_clean = pd.read_csv(
    CLEAN_PATH,
    parse_dates=["date"]
)

assert len(reloaded_clean) == len(clean)

assert (
    reloaded_clean.duplicated(
        ["store_id", "product_id", "date"]
    ).sum() == 0
)

print("Saved:", CATALOG_PATH)
print("Saved:", CLEAN_PATH)
print("CSV reload check passed")

# Data Preprocess

In [ ]:
MODEL_FEATURES = [
    "product_id",
    "lead_time_days",
    "day_of_week",
    "month",
    "units_sold_today",
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_28",
    "rolling_std_28",
]

TARGET = "target_cumulative_units_sold"
FEATURE_SCHEMA_VERSION = "features-v1"

feature_schema = {
    "version": FEATURE_SCHEMA_VERSION,
    "prediction_timing": "end_of_day_after_sales_ingestion",
    "forecast_target": TARGET,
    "model_feature_columns": MODEL_FEATURES,
    "history_days_required": 28,
    "rolling_window_rule": (
        "shift(1) then rolling(window) -- excludes current day"
    ),
    "excluded_v1_inputs": [
        "inventory_level",
        "demand_forecast",
        "units_ordered",
        "source_category",
        "region",
    ],
    "future_inventory_anomaly_inputs": [
        "product_id",
        "date",
        "units_sold",
        "units_ordered",
        "lead_time_days",
        "rolling_mean_28",
        "rolling_std_28",
    ],
}

print("Model feature count:", len(MODEL_FEATURES))
print(MODEL_FEATURES)

In [ ]:
def build_features(
    clean_history: pd.DataFrame
) -> pd.DataFrame:

    out = clean_history[
        [
            "store_id",
            "product_id",
            "date",
            "category",
            "lead_time_days",
            "units_sold",
            "units_ordered",
        ]
    ].copy()

    out = (
        out.sort_values(
            ["store_id", "product_id", "date"]
        )
        .reset_index(drop=True)
    )

    group = out.groupby(
        ["store_id", "product_id"],
        sort=False
    )["units_sold"]

    # Calendar features
    out["day_of_week"] = (
        out["date"].dt.dayofweek.astype("int64")
    )

    out["month"] = (
        out["date"].dt.month.astype("int64")
    )

    # Current observed sales (end-of-day only)
    out["units_sold_today"] = out["units_sold"]

    # Lag features
    for k in (1, 7, 14, 28):
        out[f"lag_{k}"] = group.shift(k)

    # Historical rolling averages
    for k in (7, 28):
        out[f"rolling_mean_{k}"] = group.transform(
            lambda series: (
                series.shift(1)
                .rolling(k, min_periods=k)
                .mean()
            )
        )

    # Historical standard deviation
    out["rolling_std_28"] = group.transform(
        lambda series: (
            series.shift(1)
            .rolling(28, min_periods=28)
            .std(ddof=1)
        )
    )

    return out


features = build_features(clean)

FEATURES_PATH = (
    PROCESSED_DIR / "s001_features_v1.csv"
)

features.to_csv(
    FEATURES_PATH,
    index=False,
    date_format="%Y-%m-%d",
    float_format="%.12g"
)

print("Saved:", FEATURES_PATH)

print(
    "Feature-ready rows:",
    int(features[MODEL_FEATURES].notna().all(axis=1).sum())
)

display(
    features.loc[
        features["date"].eq(pd.Timestamp("2023-11-15"))
    ].head()
)

In [ ]:
def add_future_horizon_target(
    feature_table: pd.DataFrame
) -> pd.DataFrame:

    out = feature_table.copy()

    out[TARGET] = np.nan

    out["target_end_date"] = (
        out["date"]
        + pd.to_timedelta(
            out["lead_time_days"],
            unit="D"
        )
    )

    for (_, _), indices in out.groupby(
        ["store_id", "product_id"],
        sort=False
    ).groups.items():

        group_df = (
            out.loc[indices]
            .sort_values("date")
        )

        lead_times = (
            group_df["lead_time_days"].unique()
        )

        if len(lead_times) != 1:
            raise ValueError(
                "Lead time changed inside one SKU series"
            )

        horizon = int(lead_times[0])

        # Next 1 ... L days, excluding day t
        future_days = pd.concat(
            [
                group_df["units_sold"].shift(-k)
                for k in range(1, horizon + 1)
            ],
            axis=1
        )

        # Incomplete horizons remain NaN
        target_sum = future_days.sum(
            axis=1,
            min_count=horizon
        )

        out.loc[group_df.index, TARGET] = (
            target_sum.to_numpy(dtype="float64")
        )

    return out


labeled = add_future_horizon_target(features)

model_ready = labeled.dropna(
    subset=MODEL_FEATURES + [TARGET]
).copy()

model_ready[TARGET] = (
    model_ready[TARGET].astype("int64")
)

assert (model_ready[TARGET] >= 0).all()

assert (
    model_ready["target_end_date"]
    <= clean["date"].max()
).all()

print("All raw rows:", len(clean))
print("Model-ready rows:", len(model_ready))

display(
    model_ready[
        [
            "product_id",
            "date",
            "lead_time_days",
            "target_end_date",
            TARGET,
        ]
    ].tail(8)
)

In [ ]:
TRAIN_START = pd.Timestamp("2022-01-29")
TRAIN_LAST_ORIGIN = pd.Timestamp("2023-04-09")
TRAIN_LABEL_CUTOFF = pd.Timestamp("2023-04-30")

VAL_START = pd.Timestamp("2023-05-01")
VAL_LAST_ORIGIN = pd.Timestamp("2023-08-10")
VAL_LABEL_CUTOFF = pd.Timestamp("2023-08-31")

TEST_START = pd.Timestamp("2023-09-01")
TEST_LAST_ORIGIN = pd.Timestamp("2023-12-11")
TEST_LABEL_CUTOFF = pd.Timestamp("2024-01-01")

train_mask = (
    model_ready["date"].between(
        TRAIN_START, TRAIN_LAST_ORIGIN
    )
    & model_ready["target_end_date"].le(
        TRAIN_LABEL_CUTOFF
    )
)

val_mask = (
    model_ready["date"].between(
        VAL_START, VAL_LAST_ORIGIN
    )
    & model_ready["target_end_date"].le(
        VAL_LABEL_CUTOFF
    )
)

test_mask = (
    model_ready["date"].between(
        TEST_START, TEST_LAST_ORIGIN
    )
    & model_ready["target_end_date"].le(
        TEST_LABEL_CUTOFF
    )
)

train_df = model_ready.loc[train_mask].copy()
val_df = model_ready.loc[val_mask].copy()
test_df = model_ready.loc[test_mask].copy()

# Verify disjoint splits
assert not (train_mask & val_mask).any()
assert not (train_mask & test_mask).any()
assert not (val_mask & test_mask).any()

# Verify label boundaries
assert (
    train_df["target_end_date"].max()
    <= TRAIN_LABEL_CUTOFF < VAL_START
)

assert (
    val_df["target_end_date"].max()
    <= VAL_LABEL_CUTOFF < TEST_START
)

assert (
    test_df["target_end_date"].max()
    <= TEST_LABEL_CUTOFF
)

# Mark excluded rows for traceability
model_ready["split"] = "excluded"
model_ready.loc[train_mask, "split"] = "train"
model_ready.loc[val_mask, "split"] = "validation"
model_ready.loc[test_mask, "split"] = "test"

summary = pd.DataFrame({
    "split": ["train", "validation", "test", "excluded"],
    "rows": [
        len(train_df),
        len(val_df),
        len(test_df),
        int((model_ready["split"] == "excluded").sum()),
    ],
})

display(summary)

In [ ]:
CUTOFF = test_df.loc[test_df["product_id"].eq("P0001"), "date"].min()
if pd.isna(CUTOFF):
    raise ValueError("The test split requires a valid P0001 example.")

comparison_cols = list(dict.fromkeys(
    ["store_id", "product_id"] + MODEL_FEATURES
))

def cutoff_frame(table: pd.DataFrame) -> pd.DataFrame:
    return (
        table.loc[
            table["date"].eq(CUTOFF),
            comparison_cols
        ]
        .sort_values(["store_id", "product_id"])
        .reset_index(drop=True)
    )


# Test 1: Future sales must not change today's features
reference = cutoff_frame(features)

future_corrupted = clean.copy()

future_corrupted.loc[
    future_corrupted["date"].gt(CUTOFF),
    "units_sold"
] += 100_000

pd.testing.assert_frame_equal(
    reference,
    cutoff_frame(build_features(future_corrupted))
)


# Test 2: Full history vs. cutoff-only history
cutoff_only = clean.loc[
    clean["date"].le(CUTOFF)
].copy()

pd.testing.assert_frame_equal(
    reference,
    cutoff_frame(build_features(cutoff_only))
)


# Test 3: Verify a future demand target manually
example = model_ready.loc[
    model_ready["product_id"].eq("P0001")
    & model_ready["date"].eq(CUTOFF)
].iloc[0]

manual_target = clean.loc[
    clean["product_id"].eq(example["product_id"])
    & clean["date"].gt(example["date"])
    & clean["date"].le(example["target_end_date"]),
    "units_sold",
].sum()

assert int(manual_target) == int(example[TARGET])

assert (
    example["target_end_date"]
    == example["date"]
    + pd.Timedelta(days=int(example["lead_time_days"]))
)

print("PASS: Future sales cannot affect cutoff features")
print("PASS: Cutoff-only history matches full-history features")
print("PASS: Future label matches the next L days")

In [ ]:
SUPERVISED_PATH = (
    PROCESSED_DIR / "s001_supervised_v1.csv"
)

TRAIN_PATH = SPLITS_DIR / "train_v1.csv"
VAL_PATH = SPLITS_DIR / "val_v1.csv"
TEST_PATH = SPLITS_DIR / "test_v1.csv"

SCHEMA_PATH = (
    METADATA_DIR / "feature_schema_v1.json"
)

MANIFEST_PATH = (
    METADATA_DIR / "preprocess_manifest_v1.json"
)

# Save supervised dataset and splits
for path, frame in [
    (SUPERVISED_PATH, model_ready),
    (TRAIN_PATH, train_df),
    (VAL_PATH, val_df),
    (TEST_PATH, test_df),
]:
    frame.to_csv(
        path,
        index=False,
        date_format="%Y-%m-%d",
        float_format="%.12g",
    )

# Save feature schema
SCHEMA_PATH.write_text(
    json.dumps(
        feature_schema,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

# Hash original data for reproducibility
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()

    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


manifest = {
    "dataset_version": DATA_VERSION,
    "store_id": STORE_ID,
    "source_file": RAW_PATH.name,
    "source_sha256": sha256_file(RAW_PATH),
    "product_catalog": CATALOG_PATH.name,
    "feature_schema_version": FEATURE_SCHEMA_VERSION,
    "record_count_clean": len(clean),
    "record_count_feature_and_target_complete": len(model_ready),
    "product_count": len(product_ids),
    "raw_date_min": str(clean["date"].min().date()),
    "raw_date_max": str(clean["date"].max().date()),
    "lead_time_by_category": CATEGORY_LEAD_TIMES,
    "splits": {
        "train": {
            "last_origin": str(TRAIN_LAST_ORIGIN.date()),
            "last_label": str(TRAIN_LABEL_CUTOFF.date()),
            "rows": len(train_df),
        },
        "validation": {
            "first_origin": str(VAL_START.date()),
            "last_origin": str(VAL_LAST_ORIGIN.date()),
            "last_label": str(VAL_LABEL_CUTOFF.date()),
            "rows": len(val_df),
        },
        "test": {
            "first_origin": str(TEST_START.date()),
            "last_origin": str(TEST_LAST_ORIGIN.date()),
            "last_label": str(TEST_LABEL_CUTOFF.date()),
            "rows": len(test_df),
        },
    },
    "excluded_source_columns": [
        "Inventory Level",
        "Demand Forecast",
        "Region",
        "Category (replaced by synthetic fixed category)",
    ],
    "inventory_exception_manager": (
        "Not implemented in P1/P2; retain raw sales/orders "
        "for P5 inventory replay and anomaly rules"
    ),
}

MANIFEST_PATH.write_text(
    json.dumps(
        manifest,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

for path in [
    CATALOG_PATH,
    CLEAN_PATH,
    FEATURES_PATH,
    SUPERVISED_PATH,
    TRAIN_PATH,
    VAL_PATH,
    TEST_PATH,
    SCHEMA_PATH,
    MANIFEST_PATH,
]:
    print(f"Saved {path.relative_to(PROJECT_ROOT)}")

In [ ]:
schema_on_disk = json.loads(
    SCHEMA_PATH.read_text(encoding="utf-8")
)

train_loaded = pd.read_csv(
    TRAIN_PATH,
    parse_dates=["date", "target_end_date"]
)

val_loaded = pd.read_csv(
    VAL_PATH,
    parse_dates=["date", "target_end_date"]
)

test_loaded = pd.read_csv(
    TEST_PATH,
    parse_dates=["date", "target_end_date"]
)

X_train = train_loaded[
    schema_on_disk["model_feature_columns"]
]

y_train = train_loaded[
    schema_on_disk["forecast_target"]
]

assert len(X_train) == len(y_train) == len(train_df)
assert X_train.notna().all().all()

assert "units_ordered" not in X_train.columns
assert "target_cumulative_units_sold" not in X_train.columns
assert "inventory_level" not in X_train.columns

print("Training features:", X_train.shape)
print("Validation rows:", len(val_loaded))
print("Test rows:", len(test_loaded))

display(X_train.head())

print("Phase 1 and Phase 2 completed.")

# Train

In [ ]:
from pathlib import Path
import hashlib
import json
import platform
import shutil

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import mean_absolute_error, mean_pinball_loss
from IPython.display import display

SEED = 42
np.random.seed(SEED)

DATA_DIR = PROJECT_ROOT / "data"
SPLITS_DIR = DATA_DIR / "processed" / "splits"
META_DIR = DATA_DIR / "metadata"

ARTIFACT_DIR = (
    PROJECT_ROOT / "ai" / "artifacts" / MODEL_VERSION
)
REPORT_DIR = (
    PROJECT_ROOT / "ai" / "reports" / MODEL_VERSION
)

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

for path in [
    SPLITS_DIR / "train_v1.csv",
    SPLITS_DIR / "val_v1.csv",
    SPLITS_DIR / "test_v1.csv",
    META_DIR / "feature_schema_v1.json",
    META_DIR / "product_catalog_v1.csv",
    META_DIR / "preprocess_manifest_v1.json",
]:
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing P1/P2 output: {path}"
        )

print("Project root:", PROJECT_ROOT)
print("Artifact folder:", ARTIFACT_DIR)

In [ ]:
schema = json.loads(
    (META_DIR / "feature_schema_v1.json")
    .read_text(encoding="utf-8")
)

preprocess_manifest = json.loads(
    (META_DIR / "preprocess_manifest_v1.json")
    .read_text(encoding="utf-8")
)

catalog = pd.read_csv(
    META_DIR / "product_catalog_v1.csv"
)

TRAIN_PATH = SPLITS_DIR / "train_v1.csv"
VAL_PATH = SPLITS_DIR / "val_v1.csv"
TEST_PATH = SPLITS_DIR / "test_v1.csv"


def read_split(path):
    return pd.read_csv(
        path,
        parse_dates=["date", "target_end_date"]
    )


train = read_split(TRAIN_PATH)
val = read_split(VAL_PATH)
test = read_split(TEST_PATH)

FEATURES = schema["model_feature_columns"]
TARGET = schema["forecast_target"]

CATEGORICAL_FEATURES = ["product_id"]
NUMERIC_FEATURES = [
    c for c in FEATURES
    if c not in CATEGORICAL_FEATURES
]

PRODUCT_IDS = sorted(
    catalog["product_id"].astype(str).unique().tolist()
)

PRODUCT_LEAD_TIMES = dict(zip(
    catalog["product_id"],
    catalog["lead_time_days"].astype(int)
))

HORIZONS = sorted(
    catalog["lead_time_days"]
    .astype(int)
    .unique()
    .tolist()
)

assert schema["version"] == "features-v1"
assert preprocess_manifest["store_id"] == "S001"
assert len(FEATURES) == 12
assert TARGET not in FEATURES

for forbidden in [
    "units_ordered",
    "inventory_level",
    "demand_forecast",
]:
    assert forbidden not in FEATURES

for name, frame in [
    ("train", train),
    ("validation", val),
    ("test", test),
]:
    required = FEATURES + [
        TARGET,
        "store_id",
        "date",
        "target_end_date",
    ]

    assert set(required).issubset(frame.columns), name
    assert not frame[required].isna().any().any(), name
    assert frame["store_id"].eq("S001").all(), name
    assert frame["product_id"].isin(PRODUCT_IDS).all(), name

    assert frame["target_end_date"].eq(
        frame["date"]
        + pd.to_timedelta(
            frame["lead_time_days"], unit="D"
        )
    ).all(), name

    assert frame["lead_time_days"].eq(
        frame["product_id"].map(PRODUCT_LEAD_TIMES)
    ).all(), name

    assert frame[TARGET].ge(0).all(), name

    assert not frame.duplicated(
        ["store_id", "product_id", "date"]
    ).any(), name

# Horizon labels must not cross into the next split.
assert (
    train["target_end_date"].max()
    < val["date"].min()
)

assert (
    val["target_end_date"].max()
    < test["date"].min()
)

for name, frame in [
    ("train", train),
    ("validation", val),
    ("test", test),
]:
    print(
        f"{name:>10}: {len(frame):>5} rows | "
        f"origins {frame['date'].min().date()} "
        f"to {frame['date'].max().date()} | "
        f"last label {frame['target_end_date'].max().date()}"
    )

In [ ]:
# Product ID is categorical, not ordinal numeric.
# Reuse exactly the same category mapping during P4 inference.

def prepare_X(frame):
    unknown = (
        set(frame["product_id"].astype(str))
        - set(PRODUCT_IDS)
    )

    if unknown:
        raise ValueError(
            f"Unknown products: {sorted(unknown)}"
        )

    x = frame.loc[:, FEATURES].copy()

    x["product_id"] = pd.Categorical(
        x["product_id"],
        categories=PRODUCT_IDS
    )

    for column in NUMERIC_FEATURES:
        x[column] = pd.to_numeric(
            x[column],
            errors="raise"
        ).astype("float64")

    if x.isna().any().any():
        raise ValueError("Missing feature value(s)")

    if not np.isfinite(
        x[NUMERIC_FEATURES].to_numpy()
    ).all():
        raise ValueError("Non-finite feature value(s)")

    return x


X_train = prepare_X(train)
X_val = prepare_X(val)

y_train = train[TARGET].to_numpy(dtype=float)
y_val = val[TARGET].to_numpy(dtype=float)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)

display(X_train.head())

In [ ]:
def forecast_metrics(y_true, p50, p90=None):
    y = np.asarray(y_true, dtype=float)
    median = np.asarray(p50, dtype=float)

    denom = float(np.sum(np.abs(y)))

    result = {
        "n": len(y),
        "mae": float(
            mean_absolute_error(y, median)
        ),
        "wape": (
            float(np.sum(np.abs(y - median)) / denom)
            if denom > 0 else None
        ),
        "p50_pinball": float(
            mean_pinball_loss(y, median, alpha=0.5)
        ),
    }

    if p90 is not None:
        upper = np.asarray(p90, dtype=float)

        result.update({
            "p90_pinball": float(
                mean_pinball_loss(
                    y, upper, alpha=0.9
                )
            ),
            "p90_coverage": float(
                np.mean(y <= upper)
            ),
            "mean_p90_minus_p50": float(
                np.mean(upper - median)
            ),
        })

    return result


def baseline_prediction(frame, window):
    # Recent average daily sales multiplied by lead time.
    return np.maximum(
        0.0,
        frame[f"rolling_mean_{window}"]
        .to_numpy(dtype=float)
        * frame["lead_time_days"]
        .to_numpy(dtype=float)
    )


baseline_validation = pd.DataFrame([
    {
        "model": f"rolling_{window}_x_lead_time",
        **forecast_metrics(
            y_val,
            baseline_prediction(val, window)
        ),
    }
    for window in (7, 28)
])

display(baseline_validation)

In [ ]:
# Fixed hyperparameters for V1.
# No validation/test fitting or early stopping.
# Quantiles are trained directly on cumulative L-day sales.

COMMON_PARAMS = dict(
    objective="quantile",
    n_estimators=400,
    learning_rate=0.035,
    num_leaves=15,
    max_depth=-1,
    min_child_samples=50,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1,
    deterministic=True,
    force_col_wise=True,
)

model_p50 = lgb.LGBMRegressor(
    alpha=0.5,
    **COMMON_PARAMS
)

model_p90 = lgb.LGBMRegressor(
    alpha=0.9,
    **COMMON_PARAMS
)

model_p50.fit(
    X_train,
    y_train,
    categorical_feature=CATEGORICAL_FEATURES
)

model_p90.fit(
    X_train,
    y_train,
    categorical_feature=CATEGORICAL_FEATURES
)

assert list(model_p50.feature_name_) == FEATURES
assert list(model_p90.feature_name_) == FEATURES

print("Both quantile models trained on Training split only.")

In [ ]:
val_p50_raw = np.maximum(
    0.0,
    model_p50.predict(X_val)
)

val_p90_raw = np.maximum(
    0.0,
    model_p90.predict(X_val)
)

# Avoid quantile crossing before evaluating.
val_p90_uncalibrated = np.maximum(
    val_p90_raw,
    val_p50_raw
)

validation_scores = pd.concat([
    baseline_validation,
    pd.DataFrame([{
        "model": "lightgbm_quantile_uncalibrated",
        **forecast_metrics(
            y_val,
            val_p50_raw,
            val_p90_uncalibrated
        ),
    }]),
], ignore_index=True)

display(
    validation_scores.sort_values("mae")
)

print(
    "Validation used for model comparison and calibration."
)

In [ ]:
# One-sided residual calibration.
# Score = actual demand - predicted P90.
#
# Calibration is performed separately for each lead time.
# Overlapping time-series windows mean IID conformal
# coverage guarantees do not apply.

ALPHA_P90 = 0.10


def one_sided_correction(
    y_true,
    predicted_upper,
    miscoverage=0.10
):
    errors = np.sort(
        np.asarray(y_true, dtype=float)
        - np.asarray(predicted_upper, dtype=float)
    )

    n = len(errors)

    if n == 0:
        raise ValueError(
            "Cannot calibrate from zero examples"
        )

    rank = min(
        n,
        int(np.ceil(
            (n + 1) * (1.0 - miscoverage)
        ))
    )

    # Only use nonnegative additive corrections.
    return max(0.0, float(errors[rank - 1]))


calibration_by_horizon = {}
calibration_rows = []

for horizon in HORIZONS:
    mask = (
        val["lead_time_days"].to_numpy()
        == horizon
    )

    correction = one_sided_correction(
        y_val[mask],
        val_p90_raw[mask],
        ALPHA_P90
    )

    calibration_by_horizon[str(horizon)] = correction

    calibration_rows.append({
        "lead_time_days": horizon,
        "validation_n": int(mask.sum()),
        "p90_additive_correction": correction,
    })

display(pd.DataFrame(calibration_rows))


def predict_quantiles(frame, x):
    p50 = np.maximum(
        0.0,
        model_p50.predict(x)
    )

    raw_p90 = np.maximum(
        0.0,
        model_p90.predict(x)
    )

    horizons = (
        frame["lead_time_days"]
        .to_numpy(dtype=int)
    )

    corrections = np.array([
        calibration_by_horizon[str(h)]
        for h in horizons
    ], dtype=float)

    p90 = np.maximum(
        p50,
        raw_p90 + corrections
    )

    if not (
        np.isfinite(p50).all()
        and np.isfinite(p90).all()
    ):
        raise ValueError("Invalid model output")

    assert (p50 >= 0).all()
    assert (p90 >= p50).all()

    return p50, p90, raw_p90


val_p50, val_p90, _ = predict_quantiles(
    val,
    X_val
)

print("Calibrated validation metrics (diagnostic only):")

display(pd.DataFrame([
    forecast_metrics(y_val, val_p50, val_p90)
]))

In [ ]:
# Final evaluation only.
# Do not change hyperparameters or calibration after
# inspecting Test results.

X_test = prepare_X(test)
y_test = test[TARGET].to_numpy(dtype=float)

test_p50, test_p90, test_p90_raw = predict_quantiles(
    test,
    X_test
)

test_results = pd.DataFrame([
    {
        "model": "rolling_7_x_lead_time",
        **forecast_metrics(
            y_test,
            baseline_prediction(test, 7)
        ),
    },
    {
        "model": "rolling_28_x_lead_time",
        **forecast_metrics(
            y_test,
            baseline_prediction(test, 28)
        ),
    },
    {
        "model": "lightgbm_quantile_v1",
        **forecast_metrics(
            y_test,
            test_p50,
            test_p90
        ),
    },
])

display(test_results)

# Evaluate each forecast horizon separately.
per_horizon_rows = []

for horizon in HORIZONS:
    mask = (
        test["lead_time_days"].to_numpy()
        == horizon
    )

    per_horizon_rows.append({
        "lead_time_days": int(horizon),
        **forecast_metrics(
            y_test[mask],
            test_p50[mask],
            test_p90[mask]
        ),
    })

per_horizon_results = pd.DataFrame(
    per_horizon_rows
)

display(per_horizon_results)

In [ ]:
feature_importance = pd.DataFrame({
    "feature": FEATURES,
    "p50_gain": (
        model_p50.booster_
        .feature_importance(importance_type="gain")
    ),
    "p90_gain": (
        model_p90.booster_
        .feature_importance(importance_type="gain")
    ),
})

feature_importance = (
    feature_importance
    .sort_values("p50_gain", ascending=False)
)

display(feature_importance)

# Gain-based feature importance is diagnostic,
# not evidence of causal relationships.

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as f:
        for part in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(part)

    return digest.hexdigest()


p50_path = ARTIFACT_DIR / "model_p50.joblib"
p90_path = ARTIFACT_DIR / "model_p90.joblib"

joblib.dump(model_p50, p50_path)
joblib.dump(model_p90, p90_path)

# P90 calibration configuration
calibration = {
    "method": (
        "nonnegative_one_sided_additive_residual_quantile"
    ),
    "target_coverage": 0.9,
    "calibration_split": "validation",
    "cutoff_date": str(
        val["target_end_date"].max().date()
    ),
    "correction_by_lead_time_days": calibration_by_horizon,
    "note": (
        "Overlapping windows mean IID conformal "
        "guarantees do not apply."
    ),
}

(ARTIFACT_DIR / "calibration.json").write_text(
    json.dumps(calibration, indent=2),
    encoding="utf-8"
)

# Preserve the exact feature schema and product catalog.
shutil.copy2(
    META_DIR / "feature_schema_v1.json",
    ARTIFACT_DIR / "feature_schema_v1.json"
)

shutil.copy2(
    META_DIR / "product_catalog_v1.csv",
    ARTIFACT_DIR / "product_catalog_v1.csv"
)

manifest = {
    "model_version": MODEL_VERSION,
    "store_id": "S001",
    "target": TARGET,
    "feature_schema_version": schema["version"],
    "feature_columns_in_order": FEATURES,
    "categorical_feature_columns": CATEGORICAL_FEATURES,
    "product_id_categories": PRODUCT_IDS,
    "product_lead_times": {
        k: int(v)
        for k, v in PRODUCT_LEAD_TIMES.items()
    },
    "quantiles": [0.5, 0.9],
    "model_family": "lightgbm_global_quantile",
    "training_rows": len(train),
    "training_first_origin": str(
        train["date"].min().date()
    ),
    "training_last_origin": str(
        train["date"].max().date()
    ),
    "training_last_label_date": str(
        train["target_end_date"].max().date()
    ),
    "validation_last_label_date": str(
        val["target_end_date"].max().date()
    ),
    "earliest_honest_backtest_origin": str(
        test["date"].min().date()
    ),
    "preprocess_source_sha256": (
        preprocess_manifest["source_sha256"]
    ),
    "train_csv_sha256": sha256_file(TRAIN_PATH),
    "val_csv_sha256": sha256_file(VAL_PATH),
    "test_csv_sha256": sha256_file(TEST_PATH),
    "feature_schema_sha256": sha256_file(
        ARTIFACT_DIR / "feature_schema_v1.json"
    ),
    "product_catalog_sha256": sha256_file(
        ARTIFACT_DIR / "product_catalog_v1.csv"
    ),
    "lightgbm_version": lgb.__version__,
    "scikit_learn_version": sklearn.__version__,
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "python_version": platform.python_version(),
    "random_seed": SEED,
    "training_hyperparameters": COMMON_PARAMS,
}

(ARTIFACT_DIR / "manifest.json").write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8"
)

print("Saved artifacts:", ARTIFACT_DIR)

In [ ]:
def prediction_table(frame, p50, p90, p90_raw):
    output = frame[
        [
            "store_id",
            "product_id",
            "date",
            "target_end_date",
            "lead_time_days",
            TARGET,
        ]
    ].copy()

    output["predicted_demand_p50"] = p50
    output["predicted_demand_p90"] = p90
    output["predicted_demand_p90_raw"] = p90_raw

    output["rolling_7_baseline"] = (
        baseline_prediction(frame, 7)
    )

    output["rolling_28_baseline"] = (
        baseline_prediction(frame, 28)
    )

    output["model_version"] = MODEL_VERSION

    return output


val_predictions = prediction_table(
    val,
    val_p50,
    val_p90,
    val_p90_raw
)

test_predictions = prediction_table(
    test,
    test_p50,
    test_p90,
    test_p90_raw
)

val_predictions.to_csv(
    REPORT_DIR / "validation_predictions.csv",
    index=False
)

test_predictions.to_csv(
    REPORT_DIR / "test_predictions.csv",
    index=False
)

feature_importance.to_csv(
    REPORT_DIR / "feature_importance.csv",
    index=False
)

# 将 DataFrame 中的 NaN 替换为 None，避免 json.dumps 报错
clean_validation_scores = validation_scores.replace({np.nan: None}).to_dict(orient="records")
clean_test_results = test_results.replace({np.nan: None}).to_dict(orient="records")
clean_per_horizon_results = per_horizon_results.replace({np.nan: None}).to_dict(orient="records")

report = {
    "model_version": MODEL_VERSION,
    "validation_uncalibrated_comparison": clean_validation_scores,
    "validation_calibrated_diagnostic": (
        forecast_metrics(y_val, val_p50, val_p90)
    ),
    "test_comparison": clean_test_results,
    "test_per_lead_time": clean_per_horizon_results,
    "note": (
        "Test metrics are evaluation only. "
        "Do not tune from Test results."
    ),
}

(REPORT_DIR / "evaluation_report.json").write_text(
    json.dumps(
        report,
        indent=2,
        allow_nan=False
    ),
    encoding="utf-8"
)

print("Saved evaluation reports:", REPORT_DIR)

In [ ]:
# Verify models can be reloaded without retraining.
# This prepares for the P4 FastAPI service.

loaded_p50 = joblib.load(
    ARTIFACT_DIR / "model_p50.joblib"
)

loaded_p90 = joblib.load(
    ARTIFACT_DIR / "model_p90.joblib"
)

loaded_manifest = json.loads(
    (ARTIFACT_DIR / "manifest.json")
    .read_text(encoding="utf-8")
)

loaded_calibration = json.loads(
    (ARTIFACT_DIR / "calibration.json")
    .read_text(encoding="utf-8")
)

assert loaded_manifest["feature_columns_in_order"] == FEATURES
assert loaded_manifest["product_id_categories"] == PRODUCT_IDS
assert loaded_manifest["feature_schema_version"] == schema["version"]

# Choose one valid historical Test example.
sample = test.loc[
    (test["product_id"] == "P0001")
    & (test["date"] == CUTOFF)
].copy()

assert len(sample) == 1

sample_x = prepare_X(sample)

reload_p50 = max(
    0.0,
    float(loaded_p50.predict(sample_x)[0])
)

reload_p90_raw = max(
    0.0,
    float(loaded_p90.predict(sample_x)[0])
)

horizon = int(
    sample["lead_time_days"].iloc[0]
)

reload_p90 = max(
    reload_p50,
    reload_p90_raw + float(
        loaded_calibration[
            "correction_by_lead_time_days"
        ][str(horizon)]
    )
)

sample_idx = test.index.get_loc(
    sample.index[0]
)

assert np.isclose(
    reload_p50,
    test_p50[sample_idx]
)

assert np.isclose(
    reload_p90,
    test_p90[sample_idx]
)

# Generate a sample P4 /analyze request.
# The API will derive calendar features from as_of_date.
request_features = [
    col for col in FEATURES
    if col not in {
        "product_id",
        "lead_time_days",
        "month",
        "day_of_week",
    }
]

row = sample.iloc[0]

example_request = {
    "store_id": "S001",
    "product_id": str(row["product_id"]),
    "as_of_date": str(row["date"].date()),
    "lead_time_days": horizon,
    "feature_schema_version": schema["version"],
    "input_reference": (
        f"S001/{row['product_id']}/{row['date'].date()}"
    ),
    "features": {
        name: float(row[name])
        for name in request_features
    },
}

(REPORT_DIR / "example_analyze_request.json").write_text(
    json.dumps(
        example_request,
        indent=2
    ),
    encoding="utf-8"
)

print("Reloaded forecast:")
print({
    "predicted_demand_p50": reload_p50,
    "predicted_demand_p90": reload_p90,
    "model_version": MODEL_VERSION,
})

print("P4 example request saved.")
print("P3 completed.")

In [ ]:
bundle_files = [
    "model_p50.joblib", "model_p90.joblib", "calibration.json", "manifest.json",
    "feature_schema_v1.json", "product_catalog_v1.csv",
]
(ARTIFACT_DIR / "checksums.json").write_text(
    json.dumps({name: sha256_file(ARTIFACT_DIR / name) for name in bundle_files}, indent=2),
    encoding="utf-8",
)

release_dir = PROJECT_ROOT / "release"
release_dir.mkdir(exist_ok=False)
shutil.copytree(ARTIFACT_DIR, release_dir / "artifacts" / MODEL_VERSION)
shutil.copytree(REPORT_DIR, release_dir / "reports" / MODEL_VERSION)
archive = shutil.make_archive(str(PROJECT_ROOT / MODEL_VERSION), "zip", release_dir)
files.download(archive)